# ENSANUT 2018-19 - Data Integration
This notebook integrates the five separate survey modules into a single master analytical dataset. It implements robust key harmonization (UPM, VIV_SEL, HOGAR, NUMREN) and handles column collisions between modules.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from tqdm.auto import tqdm

# Paths relative to the project root
DATA_DIR = Path("../../data")

# Standard ENSANUT keys
KEYS_IND = ['UPM', 'VIV_SEL', 'HOGAR', 'NUMREN']
KEYS_HOG = ['UPM', 'VIV_SEL', 'HOGAR']

In [3]:
def load_ensanut_csv(path):
    """Loads CSV with robust encoding and BOM stripping."""
    encodings = ['utf-8-sig', 'latin-1', 'cp1252']
    for enc in encodings:
        try:
            df = pd.read_csv(path, sep=';', encoding=enc, low_memory=False)
            # Remove any non-alphanumeric noise from column names and force UPPERCASE
            df.columns = [re.sub(r'^[^a-zA-Z0-9_]+', '', str(c)).strip().upper() for c in df.columns]
            if 'UPM' in df.columns:
                return df
        except Exception:
            continue
    raise ValueError(f"Could not load {path.name} correctly.")

def harmonize_keys(df, keys):
    """Converts keys to clean integers to ensure join compatibility."""
    for k in keys:
        if k in df.columns:
            # Handle potential str/float variations
            df[k] = pd.to_numeric(df[k].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(0).astype(int)
    return df

In [4]:
print("Starting Robust Data Integration...")

modules = {
    "Adults": "CS_ADULTOS.csv",
    "Anthropometry": "CN_ANTROPOMETRIA.csv",
    "Residents": "CS_RESIDENTES.csv",
    "Households": "CS_HOGARES.csv",
    "Biochemistry": "E18_MUESAN_DETBIO_ADUL_20200413.csv"
}

loaded = {}
for label, fname in tqdm(modules.items()):
    df = load_ensanut_csv(DATA_DIR / fname)
    loaded[label] = harmonize_keys(df, KEYS_IND + KEYS_HOG)

# Master Frame: Adults
df_final = loaded["Adults"]
print(f"Master (Adults): {df_final.shape[0]} respondents")

print("Merging Anthropometry...")
df_ant = loaded["Anthropometry"]

# ENSANUT 2018-19 splits Weight/Height into PESO1_1/PESO12_1 and TALLA4_1/TALLA15_1 based on age.
# We coalesce these into final universal variables.
print("Coalescing Weight and Height markers...")
df_ant['WEIGHT_FINAL'] = pd.to_numeric(df_ant['PESO1_1'].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(
    pd.to_numeric(df_ant['PESO12_1'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
)
df_ant['HEIGHT_FINAL'] = pd.to_numeric(df_ant['TALLA4_1'].astype(str).str.replace(',', '.', regex=False), errors='coerce').fillna(
    pd.to_numeric(df_ant['TALLA15_1'].astype(str).str.replace(',', '.', regex=False), errors='coerce')
)

cols_to_drop = [c for c in df_ant.columns if c in df_final.columns and c not in KEYS_IND]
df_final = pd.merge(df_final, df_ant.drop(columns=cols_to_drop), on=KEYS_IND, how='left')

overlap_ant = df_final['WEIGHT_FINAL'].notna().sum()
print(f"Overlap with Anthropometry (Universal): {overlap_ant}")

# Merge Residents
df_res = loaded["Residents"]
cols_to_drop = [c for c in df_res.columns if c in df_final.columns and c not in KEYS_IND]
df_final = pd.merge(df_final, df_res.drop(columns=cols_to_drop), on=KEYS_IND, how='left')

# Merge Biochemistry
df_bio = loaded["Biochemistry"]
cols_to_drop = [c for c in df_bio.columns if c in df_final.columns and c not in KEYS_IND]
df_final = pd.merge(df_final, df_bio.drop(columns=cols_to_drop), on=KEYS_IND, how='left')

# Merge Households
df_hog = loaded["Households"]
cols_to_drop = [c for c in df_hog.columns if c in df_final.columns and c not in KEYS_HOG]
df_final = pd.merge(df_final, df_hog.drop(columns=cols_to_drop), on=KEYS_HOG, how='left')

# Check for empty join disaster
if overlap_ant == 0:
    print("CRITICAL WARNING: No respondents matched between Adults and Anthropometry.")
    print("Check if UPM/VIV_SEL/HOGAR/NUMREN refer to the same individual.")

# Save
out_path = DATA_DIR / "ensanut_integrated.csv"
df_final.to_csv(out_path, index=False)
print(f"✅ Integrated data saved to {out_path}")

Starting Robust Data Integration...


  0%|          | 0/5 [00:00<?, ?it/s]

Master (Adults): 43070 respondents
Merging Anthropometry...
Coalescing Weight and Height markers...
Overlap with Anthropometry (Universal): 17164
✅ Integrated data saved to ../../data/ensanut_integrated.csv


In [5]:
import session_info

session_info.show(std_lib=True, dependencies=True)